# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [ ]:
ПРІЗВИЩЕ_ІМЯ = 'Потапова Світлана'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-31'
НОМЕР        = 16       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 16   ·   Потапова Світлана   ·   ШІ-31

Ваші вісім стовпців:
   1. ПІП
   2. Примітка
   3. Перебіг
   4. Виписка
   5. КТ
   6. Т вище 37,5
   7. Цукор2
   8. Д-дим2

Ваша пара для аналізу зв’язку: Ліжко днів та Вік

Ваше додаткове питання:
   Скільки стовпців файлу є ідентифікаторами? Як їх розпізнати автоматично, не дивлячись на назви?


In [ ]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,ПІП,Примітка,Перебіг,Виписка,КТ,"Т вище 37,5",Цукор2,Д-дим2
0,Федорюк Петро Юрійович,NaN,важкий,2022-03-10 00:00:00,20-30%,3,6.2,NaN
1,Загул Григорій Сергійович,"отримував актемра, часті носові кровотечі, втрата нюху",важкий,2022-03-16 00:00:00,30-35%,3,19.3,744
2,Бондара Лариса Павлівна,часта втрата свідомосі в стаціонарі,важкий,2022-01-18 00:00:00,70-80%,3,7.7,NaN
3,Ватаманюка Уляна Дмитрівна,панкреонекроз,важкий,2022-01-13 00:00:00,0.8,6,15.5,NaN
4,Бабійа Лариса Олегівна,NaN,важкий,2022-03-11 00:00:00,NaN,5 до гос,6.9,NaN
5,Ковалика ганна Андріївна,NaN,важкий,2021-12-26 00:00:00,30-40%,NaN,22.6,NaN
6,Височан Віктор Володимирович,"глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс",важкий,2022-05-11 00:00:00,60-65%,NaN,12.7,NaN
7,Дзюбаа Ірина Іванівна,NaN,серед важ,2022-05-28 00:00:00,до 17%,NaN,3.7,1370


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [ ]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ПІП          121 non-null    object 
 1   Примітка     45 non-null     object 
 2   Перебіг      121 non-null    object 
 3   Виписка      120 non-null    object 
 4   КТ           44 non-null     object 
 5   Т вище 37,5  78 non-null     object 
 6   Цукор2       104 non-null    float64
 7   Д-дим2       30 non-null     object 
dtypes: float64(1), object(7)
memory usage: 7.7+ KB


In [ ]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- ПІП
    непорожніх 121 з 121 | унікальних 121 | типи: str×121
    приклади: ["'Федорюк Петро Юрійович'", "'Загул Григорій Сергійович'", "'Бондара Лариса Павлівна'", "'Ватаманюка Уляна Дмитрівна'"]

--- Примітка
    непорожніх 45 з 121 | унікальних 44 | типи: str×45
    приклади: ["'отримував актемра, часті носові кровотечі, втрата нюху'", "'часта втрата свідомосі в стаціонарі'", "'панкреонекроз'", "'глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс'"]

--- Перебіг
    непорожніх 121 з 121 | унікальних 3 | типи: str×121
    приклади: ["'важкий'", "'серед важ'", "'вкрай важкий '"]

--- Виписка
    непорожніх 120 з 121 | унікальних 90 | типи: datetime×118, str×2
    приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']

--- КТ
    непорожніх 44 з 121 | унікальних 26 | типи: float×27, str×17
    приклади: ["'20-30%'", "'30-35%'", "'70-80%'", '0.8']

--- Т вище 37,5

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [ ]:
ПАСПОРТ = {
    # 'назва стовпця': ('шкала', 'обґрунтування однією фразою'),
    'ПІП': ('ідентифікатор', 'позначення конкретної особи, всі значення є унікальними'),
    'Примітка': ('номінальна', 'симптоми та супутні стани пацієнтів, які можна групувати(попередньо нормалізувавши) або порівнювати на рівність'),
    'Перебіг': ('порядкова', 'порядок за зростанням важкості стану, але відстані між ними невідомі'),
    'Виписка': ('інтервальна', 'дати з часом, мають вимірювальну різницю та чіткий порядок, але точка відліку є умовною(немає нуля)'),
    'КТ': ('відношень', 'вимірює відсоток ураження легень, є природний нуль, можна порівнювати'),
    'Т вище 37,5': ('відношень', 'стовпець містить змішані числа та текст, але за суттю це вимірювання тривалості гарячки (у днях) з природним нулем, тому обрано шкалу відношень'),
    'Цукор2': ('відношень', 'біохімічний числовий показник рівня глюкози, має природний нуль, можна порівнювати через ділення'),
    'Д-дим2': ('відношень', 'фізична концентрація маркера тромбоутворення в крові, має природний нуль, можна порівнювати через ділення')
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,ПІП,ідентифікатор,"позначення конкретної особи, всі значення є унікальними"
1,Примітка,номінальна,"симптоми та супутні стани пацієнтів, які можна групувати..."
2,Перебіг,порядкова,"порядок за зростанням важкості стану, але відстані між н..."
3,Виписка,інтервальна,"дати з часом, мають вимірювальну різницю та чіткий поряд..."
4,КТ,відношень,"вимірює відсоток ураження легень, є природний нуль, можн..."
5,"Т вище 37,5",відношень,"стовпець містить змішані числа та текст, але за суттю це..."
6,Цукор2,відношень,"біохімічний числовий показник рівня глюкози, має природн..."
7,Д-дим2,відношень,"фізична концентрація маркера тромбоутворення в крові, ма..."


In [ ]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [ ]:
def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    def clean_val(v):
        if pd.isna(v):
            return None
        if hasattr(v, 'strftime'):
            return str(v)[:10]
        if isinstance(v, (np.floating, float)):
            return round(float(v), 2)
        if isinstance(v, (np.integer, int)):
            return int(v)
        return v

    m = з.mode()
    р['мода'] = clean_val(m.iloc[0]) if not m.empty else None

    counts = з.value_counts()
    counts_dict = {clean_val(k): int(v) for k, v in counts.items()}
    р['частоти'] = counts_dict if len(counts_dict) <= 5 else dict(list(counts_dict.items())[:5])

    if ш == 'номінальна' or ш == 'ідентифікатор':
        return р

    if стовпець == 'Перебіг':
        порядок = ['серед важ', 'важкий', 'вкрай важкий ']
        cat = pd.Categorical(з, categories=порядок, ordered=True)
        codes = cat.codes[cat.codes != -1]

        if len(codes) > 0:
            p25 = int(np.percentile(codes, 25))
            p50 = int(np.percentile(codes, 50))
            p75 = int(np.percentile(codes, 75))

            р['квантиль_25'] = порядок[p25]
            р['медіана'] = порядок[p50]
            р['квантиль_75'] = порядок[p75]
    else:
        числові = pd.to_numeric(з, errors='coerce').dropna()
        if len(числові) > 0:
            р['квантиль_25'] = clean_val(числові.quantile(0.25))
            р['медіана'] = clean_val(числові.median())
            р['квантиль_75'] = clean_val(числові.quantile(0.75))

    if ш == 'порядкова':
        return р

    if 'числові' in locals() and len(числові) > 0:
        р['середнє'] = clean_val(числові.mean())
        р['std'] = clean_val(числові.std())

    if ш == 'інтервальна':
        return р

    if 'числові' in locals() and len(числові) > 0 and р['середнє'] != 0:
        р['коеф_варіації'] = round(float(р['std'] / р['середнє']), 3)

    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- ПІП  (ідентифікатор)
    {'непорожніх': 121, 'унікальних': 121, 'мода': 'Єрмолюк Григорій Павлович', 'частоти': {'Федорюк Петро Юрійович': 1, 'Загул Григорій Сергійович': 1, 'Бондара Лариса Павлівна': 1, 'Ватаманюка Уляна Дмитрівна': 1, 'Бабійа Лариса Олегівна': 1}}

--- Примітка  (номінальна)
    {'непорожніх': 45, 'унікальних': 44, 'мода': 'Поліалергетик', 'частоти': {'Поліалергетик': 2, 'отримував актемра, часті носові кровотечі, втрата нюху': 1, 'панкреонекроз': 1, 'часта втрата свідомосі в стаціонарі': 1, 'двобічний гідроторакс, на концентраторі вдома': 1}}

--- Перебіг  (порядкова)
    {'непорожніх': 121, 'унікальних': 3, 'мода': 'важкий', 'частоти': {'важкий': 90, 'серед важ': 30, 'вкрай важкий ': 1}, 'квантиль_25': 'важкий', 'медіана': 'важкий', 'квантиль_75': 'важкий'}

--- Виписка  (інтервальна)
    {'непорожніх': 120, 'унікальних': 90, 'мода': '2021-09-07', 'частоти': {'2021-11-19': 3, '2021-09-07': 3, '2021-09-11': 3, '2021-09-15': 3, '2022-06-06': 3}}

--- КТ  (відноше

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [ ]:
df_describe = df[МОЇ].copy()

df_describe['Виписка'] = pd.to_datetime(df_describe['Виписка'], errors='coerce')

for c in ['КТ', 'Т вище 37,5', 'Цукор2', 'Д-дим2']:
    df_describe[c] = pd.to_numeric(df_describe[c], errors='coerce')

stats = df_describe.describe(include='all').T
stats



,count,unique,top,freq,mean,min,25%,50%,75%,max,std
ПІП,121,121,Федорюк Петро Юрійович,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Примітка,45,44,Поліалергетик,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Перебіг,121,3,важкий,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Виписка,118,NaN,NaN,NaN,2022-01-29 18:06:06.101694976,2021-04-27 00:00:00,2021-11-16 18:00:00,2022-03-15 00:00:00,2022-04-27 18:00:00,2022-06-22 00:00:00,NaN
КТ,27.0,NaN,NaN,NaN,0.395556,0.05,0.25,0.3,0.55,0.8,0.228866
"Т вище 37,5",30.0,NaN,NaN,NaN,3.466667,1.0,3.0,3.0,4.0,7.0,1.47936
Цукор2,104.0,NaN,NaN,NaN,7.557308,3.5,4.7,6.1,9.0,22.6,3.882111
Д-дим2,29.0,NaN,NaN,NaN,1786.196897,0.25,531.0,1063.0,2546.0,5829.55,1729.928534


In [ ]:
БЕЗГЛУЗДІ = [
    # ('стовпець', 'яка статистика', значення, 'чому це число нічого не означає'),
    ('Виписка', 'mean', '2022-01-29 18:06:06.101694976', 'Середня дата в інтервальній шкалі не має фізичного змісту і не характеризує пацієнтів, не відображає реальних закономірностей чи пікових періодів виписки'),
    ('Д-дим2', 'min', 0.25, 'Мінімальне значення позбавлене сенсу, оскільки в стовпчику змішані різні одиниці вимірювання'),
    ('Т вище 37,5', 'mean', 3.466667, 'Обчислено лише по 30 числових рядках з 78 непорожніх, бо інші 48 були проігноровані через змішування чисел та тексту, тому середнє викривлене')
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Виписка,mean,2022-01-29 18:06:06.101694976,Середня дата в інтервальній шкалі не має фізичного зміст...
1,Д-дим2,min,0.25,"Мінімальне значення позбавлене сенсу, оскільки в стовпчи..."
2,"Т вище 37,5",mean,3.466667,"Обчислено лише по 30 числових рядках з 78 непорожніх, бо..."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [ ]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Ліжко днів × Вік

Ліжко днів: шкала — не у вашому списку
Вік: шкала — не у вашому списку

    Ліжко днів  Вік
0           14   55
1           22   43
2           20   54
3           20   68
4           27   65
5  9(5 у реан)   66


In [ ]:
ОБРАНА_МІРА = 'кореляція Пірсона'
ЧОМУ_САМЕ_ВОНА = 'Обидві змінні (Ліжко-дні та Вік) належать до шкал відношень, тому для оцінки лінійного зв\'язку між ними використовується коефіцієнт кореляції Пірсона'     # одне-два речення

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

# ваш код: обчисліть обрану міру

pair_data = df[['Вік', 'Ліжко днів']].copy()

pair_data['Вік'] = pd.to_numeric(pair_data['Вік'], errors='coerce')
pair_data['Ліжко днів'] = pd.to_numeric(
    pair_data['Ліжко днів'].astype(str).str.extract(r'(\d+\.?\d*)')[0],
    errors='coerce'
)

pair_clean = pair_data.dropna()
r = pair_clean['Вік'].corr(pair_clean['Ліжко днів'], method='pearson')

print(f"Коефіцієнт кореляції Пірсона для пари 'Вік' x 'Ліжко днів': {r:.3f}")


Коефіцієнт кореляції Пірсона для пари 'Вік' x 'Ліжко днів': -0.100


In [ ]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.

df_counter = df[['Перебіг', 'Ліжко днів']].dropna().copy()

df_counter['Перебіг_числовий'] = df_counter['Перебіг'].map({'серед важ': 1, 'важкий': 2, 'вкрай важкий': 3})
df_counter['Ліжко днів'] = pd.to_numeric(df_counter['Ліжко днів'].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce')

r_bad = df_counter['Перебіг_числовий'].corr(df_counter['Ліжко днів'], method='pearson')

print(f"Коефіцієнт кореляції Пірсона для пари 'Перебіг' x 'Ліжко днів': {r_bad:.3f}")


Коефіцієнт кореляції Пірсона для пари 'Перебіг' x 'Ліжко днів': 0.200


Отримане число не можна інтерпретувати, бо змінна "Перебіг" виміряна за порядковою шкалою. Числові коди (1, 2, 3) є умовними позначеннями категорій, які штучно нав'язують рівні інтервали між станами пацієнта. Для такої шкали немає сенсу рахувати середнє значення та стандартне відхилення, які використовуються у формулі коефіцієнті Пірсона. Для цієї пари потрібно використовувати рангову кореляцію Спірмена

---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [ ]:
ПІДГОТОВКА = {
    # 'стовпець': 'спосіб і коротке обґрунтування',
    'ПІП': 'не беремо у модель, оскільки це ідентифікатор особи',
    'Примітка': 'парсимо текст у декілька номінальних колонок(One-Hot кодування)',
    'Перебіг': 'кодування цілими числами із збереженням порядку',
    'Виписка': 'перехід до тривалості (розрахунок кількості днів від дати госпіталізації) та стандартизація',
    'КТ': 'стандартизація або нормалізація після вилучення текстових символів',
    'Т вище 37,5': 'нормалізація або стандартизація після вилучення текстових приміток(які можна перенести в окрему номінальну колонку)',
    'Цукор2': 'стандартизація чи нормалізація, логарифмування не потрібне бо розкид даних не великий',
    'Д-дим2': 'стандартизація, логарифмування (оскільки великий розкид значень), попередньо звівши до єдиної одиниць вимірювання'
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,ПІП,ідентифікатор,"не беремо у модель, оскільки це ідентифікатор особи"
1,Примітка,номінальна,парсимо текст у декілька номінальних колонок(One-Hot код...
2,Перебіг,порядкова,кодування цілими числами із збереженням порядку
3,Виписка,інтервальна,перехід до тривалості (розрахунок кількості днів від дат...
4,КТ,відношень,стандартизація або нормалізація після вилучення текстови...
5,"Т вище 37,5",відношень,нормалізація або стандартизація після вилучення текстови...
6,Цукор2,відношень,"стандартизація чи нормалізація, логарифмування не потріб..."
7,Д-дим2,відношень,"стандартизація, логарифмування (оскільки великий розкид ..."


In [ ]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал

df_prepared = df.copy()

mapping_перебіг = {'серед важ': 1, 'важкий': 2, 'вкрай важкий': 3}
df_prepared['Перебіг_clean'] = df_prepared['Перебіг'].astype(str).str.strip()
df_prepared['Перебіг_encoded'] = df_prepared['Перебіг_clean'].map(mapping_перебіг)

date_discharge = pd.to_datetime(df_prepared['Виписка'], errors='coerce')
date_admission = pd.to_datetime(df_prepared['Поступлення'], errors='coerce')
df_prepared['Тривалість_госпіталізації'] = (date_discharge - date_admission).dt.days

sugars = pd.to_numeric(df_prepared['Цукор2'], errors='coerce')
df_prepared['Цукор2_scaled'] = (sugars - sugars.mean()) / sugars.std()

df_prepared[['Перебіг_encoded', 'Тривалість_госпіталізації', 'Цукор2_scaled']].head()

,Перебіг_encoded,Тривалість_госпіталізації,Цукор2_scaled
0,2,14.0,-0.349631
1,2,22.0,3.024821
2,2,19.0,0.036756
3,2,20.0,2.045972
4,2,27.0,-0.169317


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [ ]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [ ]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [ ]:
ВИПРАВЛЕННЯ = [
    # (розділ, 'що не так', 'чому число неправильне', 'як правильно'),
    (
        1,
        'Некоректна інтерпретація середнього арифметичного для бінарної номінальної шкали',
        'Число 1.50 виникло через усереднення умовних кодів 1 і 2. Воно показує лише рівну частку груп (50%/50%), а не "середній" стан пацієнтів',
        'Для зручності краще перекодувати дані у 0 (помер) та 1 (вижив), тоді середнє значення (0.50) чітко вказуватиме на частку виживаності (50%)'
    ),
    (
        2,
        'Дублювання категорій номінальної шкали через різні скорочення',
        'Число 4 є хибним, оскільки воно виникло через відмінності у текстових скороченнях (ж/жін та ч/чол), які насправді позначають одну й ту саму стать',
        'Необхідно уніфікувати позначення статей, використовуючи лише один з типів скорочення'
    ),
    (
        3,
        'Застосування алфавітного сортування для порядкової шкали та відсутність уніфікації категорій',
        'Твердження про «найлегший ступінь - важкий, найважчий - середній» виникло через сортування за абеткою замість смислового порядку шкал, а число 5.9 (середній ступінь) є безглуздим через дублювання 12 неуніфікованих синонімів та обчислення середнього для умовних індексів.',
        'Спочатку уніфікувати всі варіанти написання ступеней, після чого явно задати логічний порядок категорій від легкого до критичного та обчислювати медіану чи моду'
    ),
    (
        4,
        'Втрата частини даних при перетворенні тексту в числа',
        'Показники середнього ураження 39.6%, медіани 30% та стандартного відхилення 22.9% обчислені на урізаній вибірці, оскільки функція перетворення викинула в порожні значення всі рядки з діапазонами (наприклад, «30-40%»)',
        'Спочатку очистити текст і витягнути числа (наприклад, взяти середнє для діапазонів), а вже потім рахувати показники для всього стовпчика'
    ),
    (
        5,
        'Змішування різних форматів вимірювання (дроби та відсотки) в одному стовпчику',
        'Показники мінімуму (0.9) та середнього (82.05) є викривленими, оскільки значення 90% з набору даних було єдиним записаним у відсотковому форматі та зчиталося як дріб 0.9, що штучно тягне середню сатурацію донизу',
        'Попередньо привести всі значення до єдиної шкали (помножити дробові значення на 100 або поділити цілі на 100), а вже потім обчислювати статистики'
    ),
    (
        6,
        'Обчислення коефіцієнта варіації для інтервальної шкали (дати) відносно умовного початку епохи та використання середньої дати як аналітичної метрики',
        'Коефіцієнт варіації не можна рахувати для дат, оскільки вони не мають абсолютної нульової точки. Середня дата позбавлена аналітичного сенсу',
        'Оцінювати однорідність потоку через коефіцієнт варіації кількості госпіталізацій за кожен день, а замість середньої дати аналізувати моду (пікові дні), межі дослідження (min/max) або розподіл за днями тижня'
    ),
    (
        7,
        'Використання коефіцієнта кореляції Пірсона для двох бінарних номінальних змінних',
        'Значення 0.177 є безглуздим, оскільки коефіцієнт Пірсон оцінює лише лінійний зв\'язок між кількісними неперервними величинами, а не зв\'язок між категоріальними ознаками',
        'Застосовувати спеціалізовані методи для категоріальних даних: критерій Хі-квадрат, або коефіцієнт Крамера '
    ),
    (
        8,
        'Нормалізація числових стовпчиків без попередньої перевірки їхньої варіативності',
        'Усі значення для ФЕРИТ2 перетворилися на NaN, оскільки стовпець містив лише одне значення, що призвело до ділення на нуль (X_max - X_min = 0)',
        'Перед нормалізацією необхідно перевіряти стовпчики на варіативність (що кількість унікальних значень більше 1) та вилучати константні стовпці'
    ),
    (
        9,
        'Необдумане видалення всіх рядків із пропусками',
        'Стало 0 рядків, оскільки майже в кожному пацієнта є хоча б один пропущений запис в одному із стопців. Без даних подальший аналіз неможливий..',
        'Замість повного видалення рядків слід вилучати лише ті стовпці, де занадто багато пропусків, або заповнювати відсутні значення'
    ),
    (
        10,
        'Включення у набір предикторів змінної, яка виникає тільки після настання цільової події (витік даних)',
        'Кореляція 1 є хибним артефактом, оскільки дата розтину існує виключно у померлих пацієнтів. Ця змінна не прогнозує смерть, а фіксує її факт',
        'Повністю вилучити дату розтину та інші посмертні показники з набору даних до початку побудови прогностичних моделей'
    )
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Некоректна інтерпретація середнього арифметичного для бі...,Число 1.50 виникло через усереднення умовних кодів 1 і 2...,Для зручності краще перекодувати дані у 0 (помер) та 1 (...
1,2,Дублювання категорій номінальної шкали через різні скоро...,"Число 4 є хибним, оскільки воно виникло через відмінност...","Необхідно уніфікувати позначення статей, використовуючи ..."
2,3,Застосування алфавітного сортування для порядкової шкали...,"Твердження про «найлегший ступінь - важкий, найважчий - ...","Спочатку уніфікувати всі варіанти написання ступеней, пі..."
3,4,Втрата частини даних при перетворенні тексту в числа,"Показники середнього ураження 39.6%, медіани 30% та стан...","Спочатку очистити текст і витягнути числа (наприклад, вз..."
4,5,Змішування різних форматів вимірювання (дроби та відсотк...,Показники мінімуму (0.9) та середнього (82.05) є викривл...,Попередньо привести всі значення до єдиної шкали (помнож...
5,6,Обчислення коефіцієнта варіації для інтервальної шкали (...,"Коефіцієнт варіації не можна рахувати для дат, оскільки ...",Оцінювати однорідність потоку через коефіцієнт варіації ...
6,7,Використання коефіцієнта кореляції Пірсона для двох біна...,"Значення 0.177 є безглуздим, оскільки коефіцієнт Пірсон ...",Застосовувати спеціалізовані методи для категоріальних д...
7,8,Нормалізація числових стовпчиків без попередньої перевір...,"Усі значення для ФЕРИТ2 перетворилися на NaN, оскільки с...",Перед нормалізацією необхідно перевіряти стовпчики на ва...
8,9,Необдумане видалення всіх рядків із пропусками,"Стало 0 рядків, оскільки майже в кожному пацієнта є хоча...",Замість повного видалення рядків слід вилучати лише ті с...
9,10,"Включення у набір предикторів змінної, яка виникає тільк...","Кореляція 1 є хибним артефактом, оскільки дата розтину і...",Повністю вилучити дату розтину та інші посмертні показни...


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [ ]:
# ваш код: три правильні обчислення замість хибних

print("Помилка 2: Нестандартизовані категорії статі\n")

стать_orig = df['Стать'].value_counts()
print("Неправильно (4 статі через різний формат запису):")
print(f"Кількість унікальних статей: {df['Стать'].nunique()}")
print(стать_orig.to_string())

стать_clean = df['Стать'].astype(str).str.strip().replace({'жін': 'ж', 'чол': 'ч'})
print("\nПравильно (після уніфікації синонімів):")
print(f"Кількість унікальних статей: {стать_clean.nunique()}")
print(стать_clean.value_counts().to_string())

print("-" * 60)

print("Помилка 5: Змішування відсотків та часток у SpO2\n")

s_orig = df['spO2 без дот']
print("Неправильно (з урахуванням дробу 0.9 як мінімуму):")
print(f"  середнє  : {round(s_orig.mean(), 2)}")
print(f"  мінімум  : {round(s_orig.min(), 2)}")

s_numeric = pd.to_numeric(df['spO2 без дот'], errors='coerce')
s_clean = s_numeric.apply(lambda x: x * 100 if x <= 1 else x)

print("\nПравильно (усі значення зведені до відсотків):")
print(f"  середнє  : {round(s_clean.mean(), 2)}")
print(f"  медіана  : {round(s_clean.median(), 2)}")
print(f"  мінімум  : {round(s_clean.min(), 2)}")
print(f"  максимум : {round(s_clean.max(), 2)}")

print("-" * 60)

print("Помилка 10: Витік даних через 'Дату розтину'\n")

дані_orig = df.select_dtypes('number').copy()
дані_orig['Дата розтину'] = pd.to_datetime(df['Дата розтину'], errors='coerce').astype('int64')
ціль = (df['помер/жив'] == 1).astype(int)

рейтинг_orig = {}
for c in дані_orig.columns:
    if c == 'помер/жив':
        continue
    r = дані_orig[c].corr(ціль)
    if pd.notna(r):
        рейтинг_orig[c] = abs(r)

топ_orig = pd.Series(рейтинг_orig).sort_values(ascending=False).head(5)
print("Неправильно (із посмертним показником 'Дата розтину'):")
print(топ_orig.round(3).to_string())

ознаки_clean = [c for c in df.select_dtypes('number').columns if c not in ['помер/жив', 'Дата розтину']]

рейтинг_clean = {}
for c in ознаки_clean:
    r = df[c].corr(ціль)
    if pd.notna(r):
        рейтинг_clean[c] = abs(r)

топ_clean = pd.Series(рейтинг_clean).sort_values(ascending=False).head(5)
print("\nПравильно (предиктори без витоку даних):")
print(топ_clean.round(3).to_string())

Помилка 2: Нестандартизовані категорії статі

Неправильно (4 статі через різний формат запису):
Кількість унікальних статей: 4
Стать
ж      56
ч      34
жін    19
чол    12

Правильно (після уніфікації синонімів):
Кількість унікальних статей: 2
Стать
ж    75
ч    46
------------------------------------------------------------
Помилка 5: Змішування відсотків та часток у SpO2

Неправильно (з урахуванням дробу 0.9 як мінімуму):
  середнє  : 82.05
  мінімум  : 0.9

Правильно (усі значення зведені до відсотків):
  середнє  : 82.79
  медіана  : 84.0
  мінімум  : 50.0
  максимум : 98.0
------------------------------------------------------------
Помилка 10: Витік даних через 'Дату розтину'

Неправильно (із посмертним показником 'Дата розтину'):
Дата розтину    1.000
Лім2            0.645
ІЛ6-2           0.616
Л2              0.480
Вік             0.449

Правильно (предиктори без витоку даних):
Лім2     0.645
ІЛ6-2    0.616
Л2       0.480
Вік      0.449
П/Я2     0.395


---
# Етап 7. Висновки та індивідуальне питання

In [ ]:
ВИСНОВКИ = [
    # три висновки, кожен — повне речення про ваші дані, а не загальна фраза
    '''У датасеті виявлено велику кількість пропущених та ненормалізованих даних (наприклад, показник КТ
    заповнений лише для 44 пацієнтів із 121, а стовпчик "Т вище 37,5" з кількістю днів містить змішані текстово-числові записи)''',
    ''''Очищення даних від витоку через посмертні показники виявило, що найсильнішими реальними предикторами
    летальності є рівень лімфоцитів (Лім2), інтерлейкін-6 (ІЛ6-2) та лейкоцитів (Л2)''',
    'Датасет має збалансований розподіл за результатами лікування: частка пацієнтів, які одужали, та летальних випадків становить по 50%'
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. У датасеті виявлено велику кількість пропущених та ненормалізованих даних (наприклад, показник КТ 
    заповнений лише для 44 пацієнтів із 121, а стовпчик "Т вище 37,5" з кількістю днів містить змішані текстово-числові записи)
2. 'Очищення даних від витоку через посмертні показники виявило, що найсильнішими реальними предикторами 
    летальності є рівень лімфоцитів (Лім2), інтерлейкін-6 (ІЛ6-2) та лейкоцитів (Л2)
3. Датасет має збалансований розподіл за результатами лікування: частка пацієнтів, які одужали, та летальних випадків становить по 50%


In [ ]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = '''У файлі ідентифікаторами є 3 стовпці: "номер і/х" (номер історії хвороби), "ПІП" та "Телефон".
Всі значення цих стовпців є унікальними та не мають статистичної цінності.
Автоматично виявити ідентифікатори без аналізу їхніх назв можна за такими критеріями:

1. Всі значення у стовпці є унікальними
2. Зазвичай текстовий тип даних (object/string) або цілі числа, для яких математичні обчислення не мають жодного сенсу
3. Стабільна довжина рядка або специфічні паттерни (для текстових ID), які фіксуються регулярними виразами
4. Послідовне зростання чисел з кроком 1 (для числових ID)
'''
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Скільки стовпців файлу є ідентифікаторами? Як їх розпізнати автоматично, не дивлячись на назви?

У файлі ідентифікаторами є 3 стовпці: "номер і/х" (номер історії хвороби), "ПІП" та "Телефон". 
Всі значення цих стовпців є унікальними та не мають статистичної цінності.
Автоматично виявити ідентифікатори без аналізу їхніх назв можна за такими критеріями:

1. Всі значення у стовпці є унікальними
2. Зазвичай текстовий тип даних (object/string) або цілі числа, для яких математичні обчислення не мають жодного сенсу
3. Стабільна довжина рядка або специфічні паттерни (для текстових ID), які фіксуються регулярними виразами
4. Послідовне зростання чисел з кроком 1 (для числових ID)



---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [ ]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'ні',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'синтаксис',
    'етап 4 — міра зв’язку':           'синтаксис',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'текст, перероблено',
    'етап 7 — висновки й відповідь':   'текст, перероблено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'за допомогою ШІ було згенеровано код для обчислення допустимих статистик та підготовки змінних до моделі; аналіз помилкового звіту та знаходження помилок були здійснені самостійно, але ШІ допомагав сформулювати коректні варіанти виправлень/обгрунтувань'
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [ ]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 16 готова до захисту.
Потапова Світлана, ШІ-31


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.